# Setup

```bash
conda create -n gaussian_splatting python=3.10
conda activate gaussian_splatting

pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu118

conda install -c conda-forge plyfile tqdm pip=22.3.1

pip install opencv-python joblib

# 设置 CUDA 路径
export CUDA_HOME=/usr/local/cuda-11.8
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH

pip install submodules/diff-gaussian-rasterization

pip install submodules/simple-knn 

pip install submodules/fused-ssim

```

In [7]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.6.0+cu118
True
1


In [8]:
import fused_ssim
fused_ssim

<module 'fused_ssim' from '/home/ps/anaconda3/envs/gaussian_splatting/lib/python3.10/site-packages/fused_ssim/__init__.py'>

# Running

```bash
TIME_STR=$(date +%Y%m%d_%H%M%S)
python train.py --source_path DATA/db/drjohnson \
  --model_path OUTPUT/train_drjohnson-$TIME_STR


```

1. OUTPUT/train_drjohnson-20250516_132704

```bash
conda activate gaussian_splatting
SIBR_viewers/install/bin/SIBR_gaussianViewer_app -m OUTPUT/train_drjohnson-20250516_132704

```

## Debug (launch.json)

1. 安装`Nsight Visual Studio Code Edition`插件

1. setup.py

修改 `extra_compile_args`：
```python
            extra_compile_args={
                'nvcc': [
                    '-g', '-G', '-O0',
                    '-I' +
                    os.path.join(os.path.dirname(
                        os.path.abspath(__file__)), "third_party/glm/")
                ],
                'cxx': ['-g', '-O0']
            }
```

```bash
pip uninstall diff_gaussian_rasterization
rm -rf build dist *.egg-info
pip install . -v

```


3. launch.json

```json
{
    "version": "0.2.0",
    "configurations": [
        {
            "name": "train.py",
            "type": "debugpy",
            "request": "launch",
            "program": "${workspaceFolder}/train.py",
            "args": [
                "--source_path",
                "DATA/db/drjohnson",
                "--model_path",
                "OUTPUT/train_drjohnson"
            ],
            "console": "integratedTerminal"
        },        
        {
            "name": "CUDA Debug - Attach to Python",
            "type": "cuda-gdb",
            "request": "launch",
            "program": "/home/ps/anaconda3/envs/gaussian_splatting/bin/python",
            "args": [
                "${workspaceFolder}/train.py --source_path DATA/db/drjohnson --model_path OUTPUT/train_drjohnson",                
            ],
            "initCommands": [
                "cd ${workspaceFolder}",                
            ],            
            "stopAtEntry": false,
            "console": "integratedTerminal",
            "cwd": "${workspaceFolder}",
            "miDebuggerPath": "/usr/local/cuda-11.8/bin/cuda-gdb",
            "preLaunchTask": "",
            "postDebugTask": "",
            "env": {
                "PATH": "/home/ps/anaconda3/envs/gaussian_splatting/bin:/usr/local/cuda-11.8/bin:${env:PATH}",
                "LD_LIBRARY_PATH": "/home/ps/anaconda3/envs/gaussian_splatting/lib:/usr/local/cuda-11.8/lib64:${env:LD_LIBRARY_PATH}",
                "PYTHONPATH": "${workspaceFolder}"
            },
        },
        
    ]
}
```


# Interactive Viewers

## Installation from Source （Ubuntu 22.04）

```bash
# Dependencies
proxychains4 sudo apt update
sudo apt install -y libglew-dev libassimp-dev libboost-all-dev libgtk-3-dev libopencv-dev libglfw3-dev libavdevice-dev libavcodec-dev libeigen3-dev libxxf86vm-dev libembree-dev
# Project setup
cd SIBR_viewers
export CUDA_HOME=/usr/local/cuda-11.8
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH
cmake -Bbuild . -DCMAKE_BUILD_TYPE=Release -G Ninja # add -G Ninja to build faster
cmake --build build -j24 --target install
# SIBR_viewers/install/bin/SIBR_remoteGaussian_app
# SIBR_viewers/install/bin/SIBR_gaussianViewer_app

```


nvcc报错，卸载`/usr/bin/nvcc`

```bash
dpkg -S /usr/bin/nvcc
# 如果输出像这样：
# cuda-toolkit-12-2: /usr/bin/nvcc

# 可以用以下命令卸载：
sudo apt remove cuda-toolkit-12-2

```

## Navigation in SIBR Viewers

SIBR界面提供了多种浏览场景的方法。默认情况下，系统会以FPS导航模式启动，您可以使用 W、A、S、D、Q、E 键进行摄像机平移，使用 I、K、J、L、U、O 键进行旋转。
另外，您也可以选择使用 Trackball（轨迹球）风格 的导航器（可在浮动菜单中选择）。

您还可以使用 “Snap to” 按钮跳转到数据集中某个预设摄像机的位置，或使用 “Snap to closest” 定位到离当前位置最近的摄像机。

浮动菜单还允许您调整导航速度。

此外，您可以使用 缩放修饰符（Scaling Modifier） 来控制显示的高斯点的大小，或显示初始的点云数据。
